# Authorization: Role-Based Access Control

## Overview

Understand how Graph OLAP enforces access control across different user roles. This notebook covers ownership, cross-user access, and role permissions.

**What you'll learn:**
- Analyst permissions (own resources, read access to others)
- Admin permissions (cross-user access)
- Ops permissions (inherits admin, plus ops-exclusive endpoints)
- Role boundary enforcement (strict separation)

**Prerequisites:**
- Completed: [04_advanced_falkordb_specifics.ipynb](./04_advanced_falkordb_specifics.ipynb)
- Multiple test personas configured

**Time to complete:** 25 minutes

---

**Role Hierarchy:** `Analyst < Admin < Ops` (strict superset)

| Role | Read | Write Own | Write Others | Config/Cluster/Jobs |
|------|------|-----------|--------------|---------------------|
| Analyst | All | Yes | No | No |
| Admin | All | All | All | No |
| Ops | All | All | All | Yes |

**Test Coverage:**
- Analyst ownership (9.1)
- Analyst instance access (9.2)
- Admin cross-user access (9.3)
- Ops admin inheritance (9.4)
- Role boundary enforcement (9.5)

In [ ]:
# Parameters cell - papermill will inject values here
ANALYST_USER_1 = 'analyst-alice'
ANALYST_USER_2 = 'analyst-bob'
ADMIN_USER = 'admin-user'
OPS_USER = 'ops-user'
SEEDED_MAPPING_ID = None  # Injected by papermill from fixtures
SEEDED_SNAPSHOT_ID = None  # Injected by papermill from fixtures
SEEDED_INSTANCE_ID = None  # Injected by papermill from fixtures

In [ ]:
import sys
import uuid

from graph_olap import notebook
from graph_olap.testing import TestPersona

# Wake up Starburst Galaxy cluster (auto-suspends after 5 min idle)
notebook.wake_starburst()

# Create test context as Alice (primary analyst)
ctx = notebook.test("AuthTest", persona=TestPersona.ANALYST_ALICE)

# Get clients for other personas
analyst1_client = ctx.client  # Alice
analyst2_client = ctx.with_persona(TestPersona.ANALYST_BOB)
admin_client = ctx.with_persona(TestPersona.ADMIN_CAROL)
ops_client = ctx.with_persona(TestPersona.OPS_DAVE)

print(f"Python version: {sys.version}")
print(f"Test run ID: {ctx.run_id}")
print(f"Primary persona: ANALYST_ALICE")
print(f"Additional personas: ANALYST_BOB, ADMIN_CAROL, OPS_DAVE")

## Setup and Imports

### Test Data Isolation Strategy

This notebook is **self-contained** - it creates all required test data:
1. **Base Data**: Creates mapping and instance as Alice for cross-user testing (snapshots are auto-created).
2. **Unique Prefixes**: All created resources use `AuthTest-{run_id}` prefix for easy identification.
3. **User Separation**: Each test user creates their own resources.
4. **Automatic Cleanup**: NotebookTest handles all resources with proper dependency order.

**Note**: This notebook should run in series (not parallel) with other notebooks.

In [ ]:
from graph_olap.exceptions import (
    GraphOLAPError,
    NotFoundError,
    PermissionDeniedError,
)
from graph_olap.models.mapping import EdgeDefinition, NodeDefinition, PropertyDefinition
from graph_olap_schemas import WrapperType

print("SDK imports successful")

## Create Base Test Data

Create mapping and instance as Alice for cross-user testing. Snapshots are auto-created by the platform when creating instances from mappings.

In [ ]:
# Define test node/edge for all mappings
person_node = NodeDefinition(
    label="Person",
    sql="SELECT DISTINCT id, name, age FROM bigquery.graph_olap_e2e.persons",
    primary_key={"name": "id", "type": "STRING"},
    properties=[PropertyDefinition(name="name", type="STRING"), PropertyDefinition(name="age", type="INT64")]
)

knows_edge = EdgeDefinition(
    type="KNOWS",
    from_node="Person",
    to_node="Person",
    sql="SELECT DISTINCT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships",
    from_key="from_id",
    to_key="to_id",
    properties=[PropertyDefinition(name="since", type="INT64")],
)

# Create base mapping owned by Alice using ctx (auto-tracked)
base_mapping = ctx.mapping(
    name=f"{ctx.prefix}-BaseMapping-{ctx.run_id}",
    description="Base mapping for auth testing",
    node_definitions=[person_node],
    edge_definitions=[knows_edge],
)
BASE_MAPPING_ID = base_mapping.id

print(f"Created base mapping: {base_mapping.name} (id={BASE_MAPPING_ID}, owner={base_mapping.owner_username})")

In [ ]:
# Snapshot creation is no longer needed - ctx.instance() now takes a mapping
# and auto-creates snapshots internally via create_and_wait(mapping_id=...)
# We keep BASE_SNAPSHOT_ID available from the base instance for tests that need it.
print("Skipping explicit snapshot creation (instance creation auto-creates snapshots from mappings)")

In [ ]:
# Phase 1.2 Optimization: Create base instance from mapping (auto-creates snapshot)
# Note: analyst1_instance will be created later in the ownership tests
print("Creating base instance from mapping...")

base_instance = ctx.instance(
    base_mapping,
    name=f"{ctx.prefix}-BaseInstance-{ctx.run_id}",
    wrapper_type=WrapperType.RYUGRAPH,
    timeout=180,
)
BASE_INSTANCE_ID = base_instance.id
BASE_SNAPSHOT_ID = base_instance.snapshot_id  # Auto-created by the platform

print(f"\n✓ Base instance created")
print(f"  Instance: {base_instance.name} (id={BASE_INSTANCE_ID}, status={base_instance.status})")
print(f"  Snapshot ID: {BASE_SNAPSHOT_ID} (auto-created)")
print(f"  Instance URL: {base_instance.instance_url}")

In [ ]:
# Base data created successfully
print("\nBase data created successfully:")
print(f"  Mapping: {BASE_MAPPING_ID}")
print(f"  Snapshot: {BASE_SNAPSHOT_ID} (auto-created with instance)")
print(f"  Base instance: {BASE_INSTANCE_ID} (owner=analyst_alice@e2e.local)")

## 9.1 Analyst - Resource Ownership Tests

Tests that analysts can:
- View all resources (read access to everything)
- Create their own resources
- Modify/delete only their own resources

In [ ]:
# Test 9.1.1: Analyst views all mappings
mappings = analyst1_client.mappings.list()

assert mappings is not None, "Should get mappings list"
assert len(mappings) >= 1, f"Should have at least 1 mapping, got {len(mappings)}"

# Verify can see mappings from different owners
print(f"Test 9.1.1 PASSED: Analyst can view all {len(mappings)} mapping(s)")

In [ ]:
# Test 9.1.2: Analyst views all snapshots
snapshots = analyst1_client.snapshots.list()

assert snapshots is not None, "Should get snapshots list"
assert len(snapshots) >= 1, f"Should have at least 1 snapshot, got {len(snapshots)}"

print(f"Test 9.1.2 PASSED: Analyst can view all {len(snapshots)} snapshot(s)")

In [ ]:
# Test 9.1.3: Analyst views all instances
instances = analyst1_client.instances.list()

assert instances is not None, "Should get instances list"
assert len(instances) >= 1, f"Should have at least 1 instance, got {len(instances)}"

print(f"Test 9.1.3 PASSED: Analyst can view all {len(instances)} instance(s)")

In [ ]:
# Test 9.1.4: Analyst creates own mapping - ownership assigned
test_node = NodeDefinition(
    label="AuthTestNode",
    sql="SELECT id, name FROM bigquery.graph_olap_e2e.persons",
    primary_key={"name": "id", "type": "STRING"},
    properties=[PropertyDefinition(name="name", type="STRING")]
)

test_edge = EdgeDefinition(
    type="AUTH_TEST_EDGE",
    from_node="AuthTestNode",
    to_node="AuthTestNode",
    sql="SELECT from_id, to_id FROM bigquery.graph_olap_e2e.friendships",
    from_key="from_id",
    to_key="to_id",
    properties=[],
)

# Create mapping via ctx for auto-tracking
analyst1_mapping = ctx.mapping(
    name=f"{ctx.prefix}-Analyst1Mapping-{uuid.uuid4().hex[:8]}",
    description="Created by analyst1 for auth testing",
    node_definitions=[test_node],
    edge_definitions=[test_edge],
)

assert analyst1_mapping.id is not None, "Mapping should have ID"
assert analyst1_mapping.owner_username == "analyst_alice@e2e.local", \
    f"Expected owner 'analyst_alice', got '{analyst1_mapping.owner_username}'"

analyst1_mapping_id = analyst1_mapping.id

print(f"Test 9.1.4 PASSED: Analyst created mapping (id={analyst1_mapping_id}), owner={analyst1_mapping.owner_username}")

In [ ]:
# Test 9.1.5: Analyst creates snapshot from any mapping - owns the result
# Using base mapping which belongs to Alice (same user) for this test
analyst1_snapshot_name = f"{ctx.prefix}-Analyst1Snapshot-{uuid.uuid4().hex[:8]}"
analyst1_snapshot = analyst1_client.snapshots.create(
    mapping_id=BASE_MAPPING_ID,
    name=analyst1_snapshot_name,
)

assert analyst1_snapshot.id is not None, "Snapshot should have ID"
assert analyst1_snapshot.owner_username == "analyst_alice@e2e.local", \
    f"Expected owner 'analyst_alice', got '{analyst1_snapshot.owner_username}'"

analyst1_snapshot_id = analyst1_snapshot.id

# Track for cleanup
ctx._track('snapshot', analyst1_snapshot_id, analyst1_snapshot_name)

print(f"Test 9.1.5 PASSED: Analyst created snapshot from mapping, owns result (id={analyst1_snapshot_id})")

# Note: Snapshot will be in pending status - we use base snapshot for instance creation

In [ ]:
# Test 9.1.6: Analyst creates instance from mapping - owns the result
# Create instance from base mapping (owned by Alice) - auto-creates snapshot
analyst1_instance = ctx.instance(
    base_mapping,
    name=f"{ctx.prefix}-Analyst1Instance-{uuid.uuid4().hex[:8]}",
    wrapper_type=WrapperType.RYUGRAPH,
    timeout=180,
)
analyst1_instance_id = analyst1_instance.id

# Verify analyst1_instance was created and owned correctly
analyst1_instance_check = analyst1_client.instances.get(analyst1_instance_id)
assert analyst1_instance_check.id == analyst1_instance_id, "Instance ID should match"
assert analyst1_instance_check.owner_username == "analyst_alice@e2e.local", \
    f"Expected owner 'analyst_alice', got '{analyst1_instance_check.owner_username}'"

print(f"Test 9.1.6 PASSED: Analyst created instance from mapping, owns result (id={analyst1_instance_id})")

In [ ]:
# Test 9.1.7: Analyst can update own mapping
# Note: Updating description only may not create a new version in some APIs
# We test that the update succeeds and description is changed
updated_mapping = analyst1_client.mappings.update(
    analyst1_mapping_id,
    change_description="Updated by owner",
    description="Updated description by analyst1",
)

# The update may or may not increment version depending on API semantics
# What matters is that the update succeeded (no permission error)
assert updated_mapping.id == analyst1_mapping_id, "Mapping ID should match"

print(f"Test 9.1.7 PASSED: Analyst updated own mapping (version={updated_mapping.current_version})")

In [ ]:
# Test 9.1.8: Analyst cannot update other's mapping
# analyst2 (Bob) tries to update analyst1's (Alice's) mapping
try:
    analyst2_client.mappings.update(
        analyst1_mapping_id,
        change_description="Unauthorized update attempt",
        description="Should fail",
    )
    raise AssertionError("Should have raised PermissionDeniedError")
except PermissionDeniedError:
    print("Test 9.1.8 PASSED: Analyst correctly blocked from updating other's mapping")
except GraphOLAPError as e:
    # May be a different error type depending on implementation
    if "403" in str(e) or "forbidden" in str(e).lower() or "permission" in str(e).lower():
        print(f"Test 9.1.8 PASSED: Analyst blocked from updating other's mapping ({type(e).__name__})")
    else:
        raise

In [ ]:
# Test 9.1.9: Analyst can delete own mapping (no dependencies)
# Create a fresh mapping to delete (not tracked - we'll delete it manually)
deletable_name = f"{ctx.prefix}-Deletable-{uuid.uuid4().hex[:8]}"
deletable_mapping = analyst1_client.mappings.create(
    name=deletable_name,
    description="Will be deleted",
    node_definitions=[test_node],
    edge_definitions=[test_edge],
)

deletable_id = deletable_mapping.id
analyst1_client.mappings.delete(deletable_id)

# Verify deleted
try:
    analyst1_client.mappings.get(deletable_id)
    raise AssertionError("Should have raised NotFoundError")
except NotFoundError:
    print(f"Test 9.1.9 PASSED: Analyst deleted own mapping (id={deletable_id})")

In [ ]:
# Test 9.1.10: Analyst cannot delete other's mapping
try:
    analyst2_client.mappings.delete(analyst1_mapping_id)
    raise AssertionError("Should have raised PermissionDeniedError")
except PermissionDeniedError:
    print("Test 9.1.10 PASSED: Analyst correctly blocked from deleting other's mapping")
except GraphOLAPError as e:
    if "403" in str(e) or "forbidden" in str(e).lower() or "permission" in str(e).lower():
        print(f"Test 9.1.10 PASSED: Analyst blocked from deleting other's mapping ({type(e).__name__})")
    else:
        raise

## 9.2 Analyst - Instance Access Tests

Tests for instance-specific permissions:
- Query access (read-only to any instance)
- Algorithm execution (write to own instance only)

In [ ]:
# Test: Verify dynamic instance is running
# Instance was created with create_and_wait so it's already running
analyst1_instance_ready = analyst1_client.instances.get(analyst1_instance_id)

assert analyst1_instance_ready.status == "running", \
    f"Expected status 'running', got '{analyst1_instance_ready.status}'"

print("Test 9.1.X PASSED: Dynamic instance started successfully")
print(f"  Instance ID: {analyst1_instance_ready.id}")
print(f"  Instance URL: {analyst1_instance_ready.instance_url}")
print(f"  Status: {analyst1_instance_ready.status}")

In [ ]:
# Test 9.2.1: Analyst queries any instance (read-only access)
# analyst2 (Bob) queries the base instance (owned by Alice)
conn = analyst2_client.instances.connect(BASE_INSTANCE_ID)

result = conn.query("MATCH (n) RETURN count(n) AS count")
assert result is not None, "Query should succeed"
assert result.row_count == 1, "Should get 1 row"

print(f"Test 9.2.1 PASSED: Analyst can query any instance (count={result.rows[0][0]})")

In [ ]:
# Test 9.2.2: Instance owner can run algorithm on their instance
# The base instance is owned by Alice
owner_conn = analyst1_client.instances.connect(BASE_INSTANCE_ID)

exec_result = owner_conn.algo.pagerank(
    node_label="Person",
    property_name="owner_pr",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"

# Track graph properties for cleanup via ctx
ctx._track('graph_properties', owner_conn, {'node_label': 'Person', 'property_names': ['owner_pr']})

print(f"Test 9.2.2 PASSED: Instance owner ran algorithm successfully (nodes_updated={exec_result.nodes_updated})")

In [ ]:
# Test 9.2.3: Analyst cannot run algorithm on other user's instance
# The base instance is owned by Alice
# Bob is NOT the owner, so should get 403 Permission Denied
non_owner_conn = analyst2_client.instances.connect(BASE_INSTANCE_ID)

try:
    non_owner_conn.algo.pagerank(
        node_label="Person",
        property_name="unauthorized_pr",
        edge_type="KNOWS"
    )
    raise AssertionError("Should have raised PermissionDeniedError")
except PermissionDeniedError:
    print("Test 9.2.3 PASSED: Non-owner analyst blocked from running algorithm")
except GraphOLAPError as e:
    if "403" in str(e) or "forbidden" in str(e).lower() or "permission" in str(e).lower():
        print(f"Test 9.2.3 PASSED: Non-owner analyst blocked ({type(e).__name__})")
    else:
        raise

In [ ]:
# Test 9.2.4: Analyst terminates own instance
# Create a new instance for termination testing (not tracked - we'll terminate manually)
term_instance_name = f"{ctx.prefix}-TermInst-{uuid.uuid4().hex[:8]}"
term_instance = analyst1_client.instances.create(
    mapping_id=BASE_MAPPING_ID,
    name=term_instance_name,
    wrapper_type=WrapperType.RYUGRAPH,
)
term_instance_id = term_instance.id
print(f"Created instance {term_instance_id} for termination test")

# Terminate it  
analyst1_client.instances.terminate(term_instance_id)

# Verify instance is GONE (terminate now immediately deletes)
try:
    analyst1_client.instances.get(term_instance_id)
    raise AssertionError("Instance should have been deleted after termination")
except NotFoundError:
    print(f"Test 9.2.4 PASSED: Analyst terminated own instance (id={term_instance_id}, immediately deleted)")

In [ ]:
# Test 9.2.5: Analyst cannot terminate other's instance
try:
    analyst2_client.instances.terminate(analyst1_instance_id)
    raise AssertionError("Should have raised PermissionDeniedError")
except PermissionDeniedError:
    print("Test 9.2.5 PASSED: Analyst blocked from terminating other's instance")
except GraphOLAPError as e:
    if "403" in str(e) or "forbidden" in str(e).lower() or "permission" in str(e).lower():
        print(f"Test 9.2.5 PASSED: Analyst blocked from terminating other's instance ({type(e).__name__})")
    else:
        raise

## 9.3 Admin - Cross-User Access Tests

Tests that admins can:
- Update any user's mapping
- Delete any user's resources
- Terminate any user's instance
- Run algorithms on any instance

In [ ]:
# Test 9.3.1: Admin updates any mapping
admin_updated = admin_client.mappings.update(
    analyst1_mapping_id,
    change_description="Admin update across user boundary",
    description="Updated by admin",
)

# The update may or may not increment version depending on API semantics
# What matters is that admin can update across user boundary (no permission error)
assert admin_updated.id == analyst1_mapping_id, "Mapping ID should match"

print(f"Test 9.3.1 PASSED: Admin updated analyst's mapping (version={admin_updated.current_version})")

In [ ]:
# Test 9.3.2: Admin deletes any mapping
# Create a mapping as analyst2 (Bob) that admin will delete
analyst2_mapping_name = f"{ctx.prefix}-Analyst2-{uuid.uuid4().hex[:8]}"
analyst2_mapping = analyst2_client.mappings.create(
    name=analyst2_mapping_name,
    description="To be deleted by admin",
    node_definitions=[test_node],
    edge_definitions=[test_edge],
)

analyst2_mapping_id = analyst2_mapping.id
print(f"Created mapping {analyst2_mapping_id} (owner=analyst_bob@e2e.local)")

# Admin deletes it
admin_client.mappings.delete(analyst2_mapping_id)

# Verify deleted
try:
    admin_client.mappings.get(analyst2_mapping_id)
    raise AssertionError("Should have raised NotFoundError")
except NotFoundError:
    print(f"Test 9.3.2 PASSED: Admin deleted analyst2's mapping (id={analyst2_mapping_id})")

In [ ]:
# Test 9.3.3: Admin deletes any snapshot
# Create snapshot as analyst2 (Bob) (use create_and_wait to ensure it exists)
analyst2_snapshot_name = f"{ctx.prefix}-A2Snap-{uuid.uuid4().hex[:8]}"
analyst2_snapshot = analyst2_client.snapshots.create_and_wait(
    mapping_id=BASE_MAPPING_ID,
    name=analyst2_snapshot_name,
    timeout=180,
)

analyst2_snapshot_id = analyst2_snapshot.id
print(f"Created snapshot {analyst2_snapshot_id} (owner=analyst_bob@e2e.local, status={analyst2_snapshot.status})")

# Admin deletes it
admin_client.snapshots.delete(analyst2_snapshot_id)

# Verify deleted
try:
    admin_client.snapshots.get(analyst2_snapshot_id)
    raise AssertionError("Should have raised NotFoundError")
except NotFoundError:
    print(f"Test 9.3.3 PASSED: Admin deleted analyst2's snapshot (id={analyst2_snapshot_id})")

In [ ]:
# Test 9.3.4: Admin terminates any instance
# Create instance from base mapping (analyst2/Bob owns it, admin terminates it)
analyst2_instance_name = f"{ctx.prefix}-A2Inst-{uuid.uuid4().hex[:8]}"
analyst2_instance = analyst2_client.instances.create(
    mapping_id=BASE_MAPPING_ID,
    name=analyst2_instance_name,
    wrapper_type=WrapperType.RYUGRAPH,
)

analyst2_instance_id = analyst2_instance.id
print(f"Created instance {analyst2_instance_id} owned by analyst_bob@e2e.local")

# Admin terminates it (doesn't need to be running to test permission)
admin_client.instances.terminate(analyst2_instance_id)

# Verify instance is GONE (terminate now immediately deletes)
try:
    admin_client.instances.get(analyst2_instance_id)
    raise AssertionError("Instance should have been deleted after termination")
except NotFoundError:
    print(f"Test 9.3.4 PASSED: Admin terminated analyst2's instance (id={analyst2_instance_id}, immediately deleted)")

In [ ]:
# Test 9.3.5: Admin runs algorithm on any instance
admin_conn = admin_client.instances.connect(BASE_INSTANCE_ID)

exec_result = admin_conn.algo.pagerank(
    node_label="Person",
    property_name="admin_cross_pr",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"

# Track graph properties for cleanup
ctx._track('graph_properties', admin_conn, {'node_label': 'Person', 'property_names': ['admin_cross_pr']})

print(f"Test 9.3.5 PASSED: Admin ran algorithm on base instance (nodes_updated={exec_result.nodes_updated})")

In [ ]:
# Test 9.3.6: Admin views export queue
# Note: This requires the exports endpoint to be available in SDK
# If not available, we test that admin can at least list snapshots with export status
try:
    exports = admin_client.snapshots.list()  # Admin should have full access
    print("Test 9.3.6 PASSED: Admin can view resources (proxy for export queue access)")
except Exception as e:
    print(f"Test 9.3.6 SKIPPED: Export queue endpoint not in SDK ({e})")

In [ ]:
# Test 9.3.7: Admin retries failed export (SKIPPED - requires internal API)
# 
# This test would verify that admins can retry failed snapshots.
# However, it requires the internal API endpoint to be available.
# In the current E2E setup, this endpoint may not be exposed.

print("Test 9.3.7 SKIPPED: Admin retry test requires internal API (not exposed in E2E)")

## 9.4 Ops Role - Admin Inheritance Tests

Tests that ops users inherit all admin permissions:
- Ops can update any mapping (ownership bypass via admin inheritance)
- Ops can delete any snapshot (ownership bypass via admin inheritance)
- Ops can bulk delete (inherits admin endpoint access)

In [ ]:
# Test 9.4.1: Ops updates any mapping (inherits admin)
ops_updated = ops_client.mappings.update(
    analyst1_mapping_id,
    change_description="Ops update via admin inheritance",
    description="Updated by ops user",
)

assert ops_updated.id == analyst1_mapping_id, "Mapping ID should match"

print(f"Test 9.4.1 PASSED: Ops updated analyst's mapping (version={ops_updated.current_version})")

In [ ]:
# Test 9.4.2: Ops deletes any snapshot (inherits admin)
# Create a snapshot as analyst2 (Bob) that ops will delete
ops_test_snap_name = f"{ctx.prefix}-OpsTestSnap-{uuid.uuid4().hex[:8]}"
ops_test_snapshot = analyst2_client.snapshots.create_and_wait(
    mapping_id=BASE_MAPPING_ID,
    name=ops_test_snap_name,
    timeout=180,
)

ops_test_snapshot_id = ops_test_snapshot.id
print(f"Created snapshot {ops_test_snapshot_id} (owner=analyst_bob@e2e.local)")

# Ops deletes it
ops_client.snapshots.delete(ops_test_snapshot_id)

# Verify deleted
try:
    ops_client.snapshots.get(ops_test_snapshot_id)
    raise AssertionError("Should have raised NotFoundError")
except NotFoundError:
    print(f"Test 9.4.2 PASSED: Ops deleted analyst2's snapshot (id={ops_test_snapshot_id})")

In [ ]:
# Test 9.4.3: Ops terminates any instance (inherits admin)
# Create instance as analyst2 that ops will terminate
ops_term_name = f"{ctx.prefix}-OpsTermInst-{uuid.uuid4().hex[:8]}"
ops_term_instance = analyst2_client.instances.create(
    mapping_id=BASE_MAPPING_ID,
    name=ops_term_name,
    wrapper_type=WrapperType.RYUGRAPH,
)

ops_term_id = ops_term_instance.id
print(f"Created instance {ops_term_id} (owner=analyst_bob@e2e.local)")

# Ops terminates it
ops_client.instances.terminate(ops_term_id)

# Verify deleted
try:
    ops_client.instances.get(ops_term_id)
    raise AssertionError("Instance should have been deleted")
except NotFoundError:
    print(f"Test 9.4.3 PASSED: Ops terminated analyst2's instance (id={ops_term_id})")

## 9.5 Role Boundary Tests

Tests strict role boundaries:
- Admin CANNOT access ops-only endpoints (/api/ops/*)
- Analyst CANNOT access admin or ops endpoints

In [ ]:
# Test 9.5.1: Admin CANNOT access ops config endpoints
try:
    admin_client.ops.get_lifecycle_config()
    raise AssertionError("Should have raised PermissionDeniedError")
except PermissionDeniedError:
    print("Test 9.5.1 PASSED: Admin correctly blocked from ops config endpoints")
except GraphOLAPError as e:
    if isinstance(e, PermissionDeniedError) or "ops" in str(e).lower() or "403" in str(e):
        print(f"Test 9.5.1 PASSED: Admin blocked from config endpoints ({type(e).__name__})")
    else:
        raise

In [ ]:
# Test 9.5.2: Admin CANNOT access cluster health
try:
    admin_client.ops.get_cluster_health()
    raise AssertionError("Should have raised PermissionDeniedError")
except PermissionDeniedError:
    print("Test 9.5.2 PASSED: Admin correctly blocked from cluster health endpoint")
except GraphOLAPError as e:
    if isinstance(e, PermissionDeniedError) or "ops" in str(e).lower() or "403" in str(e):
        print(f"Test 9.5.2 PASSED: Admin blocked from cluster health ({type(e).__name__})")
    else:
        raise

In [ ]:
# Test 9.5.3: Admin CANNOT trigger background jobs
try:
    admin_client.ops.trigger_job(job_name="reconciliation", reason="admin-boundary-test")
    raise AssertionError("Should have raised PermissionDeniedError")
except PermissionDeniedError:
    print("Test 9.5.3 PASSED: Admin correctly blocked from triggering ops jobs")
except GraphOLAPError as e:
    if isinstance(e, PermissionDeniedError) or "ops" in str(e).lower() or "403" in str(e):
        print(f"Test 9.5.3 PASSED: Admin blocked from ops jobs ({type(e).__name__})")
    else:
        raise

In [ ]:
print("\n" + "="*60)
print("AUTHORIZATION E2E TESTS COMPLETED!")
print("="*60)
print("\nValidated:")
print("  9.1 Analyst Ownership:")
print("    - 9.1.1: Analyst views all mappings")
print("    - 9.1.2: Analyst views all snapshots")
print("    - 9.1.3: Analyst views all instances")
print("    - 9.1.4: Analyst creates mapping, owns it")
print("    - 9.1.5: Analyst creates snapshot from any mapping, owns it")
print("    - 9.1.6: Analyst creates instance from mapping, owns it")
print("    - 9.1.7: Analyst updates own mapping")
print("    - 9.1.8: Analyst cannot update other's mapping")
print("    - 9.1.9: Analyst deletes own mapping")
print("    - 9.1.10: Analyst cannot delete other's mapping")
print("  9.2 Analyst Instance Access:")
print("    - 9.2.1: Analyst queries any instance")
print("    - 9.2.2: Instance owner runs algorithm on own instance")
print("    - 9.2.3: Non-owner analyst blocked from running algorithm")
print("    - 9.2.4: Analyst terminates own instance")
print("    - 9.2.5: Analyst cannot terminate other's instance")
print("  9.3 Admin Cross-User Access:")
print("    - 9.3.1: Admin updates any mapping")
print("    - 9.3.2: Admin deletes any mapping")
print("    - 9.3.3: Admin deletes any snapshot")
print("    - 9.3.4: Admin terminates any instance")
print("    - 9.3.5: Admin runs algorithm on any instance")
print("    - 9.3.6: Admin views export queue (proxy)")
print("    - 9.3.7: SKIPPED (requires internal API)")
print("  9.4 Ops Role - Admin Inheritance:")
print("    - 9.4.1: Ops updates any mapping (inherits admin)")
print("    - 9.4.2: Ops deletes any snapshot (inherits admin)")
print("    - 9.4.3: Ops terminates any instance (inherits admin)")
print("  9.5 Role Boundary Tests:")
print("    - 9.5.1: Admin blocked from ops config endpoints")
print("    - 9.5.2: Admin blocked from cluster health")
print("    - 9.5.3: Admin blocked from triggering ops jobs")
print("\nAll test resources will be cleaned up automatically via atexit")